In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import confusion_matrix, classification_report
# Optional plotting and optional LightGBM
try:
    import seaborn as sns
except Exception:
    sns = None
# Optional LightGBM: used if installed for an ensemble
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# Option: apply multiplicative calibration (learn factor on training preds vs actuals)
APPLY_MULTIPLICATIVE_CALIBRATION = True

# Optional Excel engine check: provide actionable message when missing
try:
    import openpyxl  # pandas uses this to read .xlsx files
except Exception:
    print("Missing optional dependency 'openpyxl'. To read Excel files, install it with:\n\n  pip install openpyxl\n\nor with conda:\n\n  conda install -c conda-forge openpyxl")
    # Do not raise here so the notebook can still be inspected; reading will fail later if used

def main(file_path):
    # ---------------------------------------------------------
    # 1. LOAD AND CLEAN DATA
    # ---------------------------------------------------------
    #file_path = 'problem_data.xlsx - in.csv'  # Update with your actual file name
    print("Loading data...")
    
    # Load data
    df = pd.read_excel(file_path)
    
    # Ensure correct data types
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['Reading'] = pd.to_numeric(df['Reading'], errors='coerce')
    
    # Drop invalid rows
    df = df.dropna(subset=['timestamp', 'Reading'])
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['timestamp'], keep='first')
    
    # Sort by time
    df = df.sort_values('timestamp').set_index('timestamp')
    
    # Resample to 5-second intervals and interpolate missing values
    # This ensures a consistent time grid for the model
    df_resampled = df.resample('5S').interpolate(method='linear')
    
    print(f"Data cleaned. Total rows: {len(df_resampled)}")

    # ---------------------------------------------------------
    # 2. ANOMALY DETECTION (IQR Method)
    # ---------------------------------------------------------
    print("\n--- Anomaly Detection ---")
    Q1 = df_resampled['Reading'].quantile(0.25)
    Q3 = df_resampled['Reading'].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    anomalies = df_resampled[(df_resampled['Reading'] < lower_bound) | (df_resampled['Reading'] > upper_bound)]
    
    print(f"IQR Lower Bound: {lower_bound:.2f}")
    print(f"IQR Upper Bound: {upper_bound:.2f}")
    print(f"Number of anomalies detected: {len(anomalies)}")
    
    # ---------------------------------------------------------
    # 3. QUARTILE ANALYSIS
    # ---------------------------------------------------------
    print("\n--- Quartile Analysis ---")
    stats = df_resampled['Reading'].describe()
    print(stats)

    # ---------------------------------------------------------
    # 4. FORECASTING MODEL DEVELOPMENT
    # ---------------------------------------------------------
    print("\n--- Training Forecast Model ---")
    
    # Goal: Forecast 3 hours in advance.
    # 3 hours = 3 * 60 * 60 seconds = 10800 seconds
    # Step size = 5 seconds
    # Steps ahead = 10800 / 5 = 2160 steps
    FORECAST_STEPS = 2160
    
    df_model = df_resampled.copy()
    
    # Create Target: The reading 3 hours into the future
    df_model['Target'] = df_model['Reading'].shift(-FORECAST_STEPS)
    
    # Feature Engineering
    # We use past values to predict the future target
    df_model['Lag_1'] = df_model['Reading'].shift(1)       # Immediate previous
    df_model['Lag_12'] = df_model['Reading'].shift(12)     # 1 minute ago
    df_model['Lag_720'] = df_model['Reading'].shift(720)   # 1 hour ago
    df_model['Lag_2160'] = df_model['Reading'].shift(2160) # 3 hours ago (same seasonality)
    df_model['Rolling_Mean_12'] = df_model['Reading'].rolling(window=12).mean()
    
    # Time features
    df_model['Hour'] = df_model.index.hour
    df_model['Minute'] = df_model.index.minute
    
    # Remove rows with NaN (caused by shifting/lagging) for training
    df_train_valid = df_model.dropna()
    
    # Define Features and Target
    features = ['Reading', 'Lag_1', 'Lag_12', 'Lag_720', 'Lag_2160', 'Rolling_Mean_12', 'Hour', 'Minute']
    X = df_train_valid[features]
    y = df_train_valid['Target']
    
    # Train/Test Split (Time-based split, keeping last 20% for validation)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Remove outliers from TRAINING data only (outside 10th-90th percentiles)
    p10 = X_train['Reading'].quantile(0.10)
    p90 = X_train['Reading'].quantile(0.90)
    train_mask = (X_train['Reading'] >= p10) & (X_train['Reading'] <= p90)
    removed_count = len(X_train) - int(train_mask.sum())
    print(f"Removing {removed_count} rows from training (outside 10th-90th percentiles)")
    X_train = X_train[train_mask]
    y_train = y_train.loc[X_train.index]

    # Add longer-seasonality lags and rolling features if they don't already exist
    for lag in [4320, 8640, 17280]:
        col = f'Lag_{lag}'
        if col not in df_model.columns:
            df_model[col] = df_model['Reading'].shift(lag)
    # Rebuild train/test after ensuring features exist
    df_train_valid = df_model.dropna()
    features = ['Reading', 'Lag_1', 'Lag_12', 'Lag_60' if 'Lag_60' in df_model.columns else 'Lag_1', 'Lag_720', 'Lag_2160', 'Rolling_Mean_12', 'Hour', 'Minute']
    # Add explicit new features
    extra = ['Lag_4320', 'Lag_8640', 'Lag_17280']
    for e in extra:
        if e in df_model.columns and e not in features:
            features.append(e)
    # rolling 60s mean/std
    df_model['Rolling_Mean_60'] = df_model['Reading'].rolling(window=60).mean()
    df_model['Rolling_Std_60'] = df_model['Reading'].rolling(window=60).std()
    if 'Rolling_Mean_60' not in features:
        features.extend(['Rolling_Mean_60', 'Rolling_Std_60'])
    # time cycle
    seconds = df_model.index.hour * 3600 + df_model.index.minute * 60 + df_model.index.second
    df_model['sin_day'] = np.sin(2 * np.pi * seconds / 86400)
    df_model['cos_day'] = np.cos(2 * np.pi * seconds / 86400)
    df_model['dayofweek'] = df_model.index.dayofweek
    for tcol in ['sin_day', 'cos_day', 'dayofweek']:
        if tcol not in features:
            features.append(tcol)

    # Recreate X/y from df_model and re-split to ensure features present in X_test
    df_train_valid = df_model.dropna()
    X = df_train_valid[features]
    y = df_train_valid['Target']
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Apply the same 10-90% training filter (recompute on X_train)
    p10 = X_train['Reading'].quantile(0.10)
    p90 = X_train['Reading'].quantile(0.90)
    train_mask = (X_train['Reading'] >= p10) & (X_train['Reading'] <= p90)
    removed_count = len(X_train) - int(train_mask.sum())
    if removed_count > 0:
        print(f"(After feature expansion) removing {removed_count} rows from training")
    X_train = X_train[train_mask]
    y_train = y_train.loc[X_train.index]

    # Fill NaNs created by rolling features
    X_train = X_train.fillna(method='ffill').fillna(method='bfill')
    X_test = X_test.fillna(method='ffill').fillna(method='bfill')

    # Log-transform the target to stabilize variance (train on log1p)
    y_train_log = np.log1p(y_train)

    # Train HistGradientBoosting on log-target
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train_log)

    # Optionally train LightGBM and ensemble (if available)
    preds_log_hgb = model.predict(X_test)
    if LGB_AVAILABLE:
        lgb_model = lgb.LGBMRegressor(random_state=42)
        lgb_model.fit(X_train, y_train_log)
        preds_log_lgb = lgb_model.predict(X_test)
        preds_log = 0.5 * (preds_log_hgb + preds_log_lgb)
    else:
        preds_log = preds_log_hgb

    # Invert log transform for predictions
    predictions = np.expm1(preds_log)

    # Bias correction using training set residuals (on original scale)
    try:
        preds_train_log = model.predict(X_train)
        preds_train = np.expm1(preds_train_log)
        bias_train = float(np.mean(preds_train - y_train))
        print(f"Applying additive bias correction of {bias_train:.4f} (units) based on training set")
        predictions = predictions - bias_train

        # Multiplicative calibration: learn average ratio preds/actuals on training (avoid zeros)
        multiplicative_factor = None
        try:
            mask_pos = (y_train.values != 0)
            if np.any(mask_pos):
                ratios = preds_train[mask_pos] / y_train.values[mask_pos]
                multiplicative_factor = float(np.nanmean(ratios))
                if APPLY_MULTIPLICATIVE_CALIBRATION and multiplicative_factor not in (None, 0):
                    print(f"Applying multiplicative calibration factor {multiplicative_factor:.4f} (dividing predictions by factor)")
                    predictions = predictions / multiplicative_factor
        except Exception as _:
            multiplicative_factor = None
    except Exception as e:
        bias_train = 0.0
        multiplicative_factor = None
        print(f"Could not compute/apply bias correction: {e}")

    # Clip negative predictions to zero (predictions should not be negative)
    predictions = np.clip(predictions, 0, None)

    # Evaluate (regression) on original scale
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    # Percent errors (MAPE) and percentage bias (MPE)
    y_vals = y_test.values
    preds = predictions
    zero_actuals_mask = (y_vals == 0)
    zero_count = int(np.sum(zero_actuals_mask))
    with np.errstate(divide='ignore', invalid='ignore'):
        perc_errors = np.abs((y_vals - preds) / y_vals) * 100
        perc_errors[zero_actuals_mask] = np.nan
        mpe_vals = ((preds - y_vals) / y_vals) * 100
        mpe_vals[zero_actuals_mask] = np.nan

    mape = float(np.nanmean(perc_errors)) if not np.all(np.isnan(perc_errors)) else np.nan
    mpe = float(np.nanmean(mpe_vals)) if not np.all(np.isnan(mpe_vals)) else np.nan
    # SMAPE (symmetric MAPE)
    with np.errstate(divide='ignore', invalid='ignore'):
        denom = (np.abs(y_vals) + np.abs(preds))
        smape_arr = 2 * np.abs(preds - y_vals) / denom
        smape_arr[denom == 0] = np.nan
        SMAPE = float(np.nanmean(smape_arr)) * 100
    # Median absolute percentage error (ignore zero actuals)
    with np.errstate(divide='ignore', invalid='ignore'):
        ape_arr = np.abs((y_vals - preds) / y_vals) * 100
        ape_arr[zero_actuals_mask] = np.nan
        median_ape = float(np.nanmedian(ape_arr))

    mean_error = float(np.mean(preds - y_vals))
    over_count = int(np.sum(preds > y_vals))
    under_count = int(np.sum(preds < y_vals))

    print(f"Model Performance -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape if not np.isnan(mape) else 'NA'}%")
    print(f"Mean Percentage Error (MPE): {mpe if not np.isnan(mpe) else 'NA'}%")
    print(f"Mean Error (bias in units): {mean_error:.4f} (over: {over_count}, under: {under_count}, zeros_in_actuals: {zero_count})")

    # Save evaluation metrics to JSON for later inspection
    try:
        import json
        metrics = {
            'MAE': mae,
            'RMSE': rmse,
            'MAPE_percent': None if np.isnan(mape) else mape,
            'MPE_percent': None if np.isnan(mpe) else mpe,
            'SMAPE_percent': SMAPE,
            'Median_APE_percent': median_ape,
            'Mean_Error_units': mean_error,
            'Over_Count': over_count,
            'Under_Count': under_count,
            'Zero_Actuals_Count': zero_count,
            'Test_Size': int(len(y_vals)),
            'Bias_train_units': bias_train,
            'Multiplicative_Factor': multiplicative_factor
        }
        with open('evaluation_metrics.json', 'w') as mf:
            json.dump(metrics, mf, indent=2)
        print("Evaluation metrics saved to evaluation_metrics.json")
    except Exception as e:
        print(f"Could not save evaluation metrics to file: {e}")

    # ---------------------------------------------------------
    # Classification for Confusion Matrix (directional)
    # ---------------------------------------------------------
    # Many forecasting problems are also evaluated by direction (up/down/flat).
    # We'll create 3 classes: -1 (down), 0 (flat), 1 (up) based on whether the
    # target 3-hours-ahead is lower/same/higher than the current reading.
    def direction_class(actual, current, tol=1e-8):
        diff = actual - current
        if diff > tol:
            return 1
        elif diff < -tol:
            return -1
        else:
            return 0

    # Current reading available in X_test as feature 'Reading'
    current_readings = X_test['Reading']

    # True direction classes
    y_true_dir = [direction_class(a, c) for a, c in zip(y_test.values, current_readings.values)]

    # Predicted direction classes
    y_pred_dir = [direction_class(p, c) for p, c in zip(predictions, current_readings.values)]

    # Confusion matrix and classification report
    labels = [-1, 0, 1]
    cm = confusion_matrix(y_true_dir, y_pred_dir, labels=labels)
    print('\nConfusion Matrix (rows=true, cols=pred) for classes [-1,0,1]:')
    print(cm)
    print('\nClassification Report:')
    print(classification_report(y_true_dir, y_pred_dir, labels=labels))

    # ---------------------------------------------------------
    # 5. GENERATE FUTURE FORECAST
    # ---------------------------------------------------------
    print("\n--- Generating Future Forecast ---")
    
    # To predict the future (after the dataset ends), we need the most recent inputs.
    # We need to construct the features for the timestamps [End - 3hrs] to [End].
    # These inputs will predict targets for [End] to [End + 3hrs].
    
    # Get the feature set for the entire original dataset (including the tail where Target is NaN)
    # We reconstruct features on the full resampled df
    df_full_features = df_resampled.copy()
    # Recreate all features used by model for the full dataset tail
    df_full_features['Lag_1'] = df_full_features['Reading'].shift(1)
    df_full_features['Lag_12'] = df_full_features['Reading'].shift(12)
    df_full_features['Lag_720'] = df_full_features['Reading'].shift(720)
    df_full_features['Lag_2160'] = df_full_features['Reading'].shift(2160)
    for lag in [4320, 8640, 17280]:
        df_full_features[f'Lag_{lag}'] = df_full_features['Reading'].shift(lag)
    df_full_features['Rolling_Mean_12'] = df_full_features['Reading'].rolling(window=12).mean()
    df_full_features['Rolling_Mean_60'] = df_full_features['Reading'].rolling(window=60).mean()
    df_full_features['Rolling_Std_60'] = df_full_features['Reading'].rolling(window=60).std()
    df_full_features['Hour'] = df_full_features.index.hour
    df_full_features['Minute'] = df_full_features.index.minute
    seconds = df_full_features.index.hour * 3600 + df_full_features.index.minute * 60 + df_full_features.index.second
    df_full_features['sin_day'] = np.sin(2 * np.pi * seconds / 86400)
    df_full_features['cos_day'] = np.cos(2 * np.pi * seconds / 86400)
    df_full_features['dayofweek'] = df_full_features.index.dayofweek
    
    # Select the last 2160 rows (last 3 hours of data)
    # These rows contain the inputs needed to predict the NEXT 3 hours
    X_future = df_full_features[features].iloc[-FORECAST_STEPS:]
    X_future = X_future.fillna(method='ffill').fillna(method='bfill')

    # Predict on log-scale and invert
    preds_log_future_hgb = model.predict(X_future)
    if LGB_AVAILABLE:
        preds_log_future_lgb = lgb_model.predict(X_future)
        preds_log_future = 0.5 * (preds_log_future_hgb + preds_log_future_lgb)
    else:
        preds_log_future = preds_log_future_hgb
    future_preds = np.expm1(preds_log_future)
    # Apply additive bias correction learned from training
    try:
        future_preds = future_preds - bias_train
    except NameError:
        pass
    # Apply multiplicative calibration if available
    try:
        if APPLY_MULTIPLICATIVE_CALIBRATION and multiplicative_factor not in (None, 0):
            future_preds = future_preds / multiplicative_factor
    except NameError:
        pass
    # Clip negatives
    future_preds = np.clip(future_preds, 0, None)
    
    # Create timestamps for predictions
    last_timestamp = df_resampled.index[-1]
    future_dates = pd.date_range(start=last_timestamp + pd.Timedelta(seconds=5), 
                                 periods=FORECAST_STEPS, 
                                 freq='5S')
    
    # Save to DataFrame
    df_forecast = pd.DataFrame({
        'timestamp': future_dates,
        'Predicted_Reading': future_preds
    })
    
    print("Forecast generated for the next 3 hours.")
    print(df_forecast.head())
    
    # Save to CSV
    output_filename = 'future_predictions_generated.csv'
    df_forecast.to_csv(output_filename, index=False)
    print(f"Forecast saved to {output_filename}")


file_path = r"C:\Z\Test\Talentica\problem_data.xlsx"
main(file_path)

Loading data...
Data cleaned. Total rows: 518400

--- Anomaly Detection ---
IQR Lower Bound: -45.32
IQR Upper Bound: 193.90
Number of anomalies detected: 0

--- Quartile Analysis ---
count    518400.000000
mean         73.126716
std          39.519886
min           0.000000
25%          44.387786
50%          86.940680
75%         104.193719
max         149.889720
Name: Reading, dtype: float64

--- Training Forecast Model ---
Removing 82254 rows from training (outside 10th-90th percentiles)
Data cleaned. Total rows: 518400

--- Anomaly Detection ---
IQR Lower Bound: -45.32
IQR Upper Bound: 193.90
Number of anomalies detected: 0

--- Quartile Analysis ---
count    518400.000000
mean         73.126716
std          39.519886
min           0.000000
25%          44.387786
50%          86.940680
75%         104.193719
max         149.889720
Name: Reading, dtype: float64

--- Training Forecast Model ---
Removing 82254 rows from training (outside 10th-90th percentiles)
(After feature expansion

c:\Python\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-sc

              precision    recall  f1-score   support

          -1       0.58      0.92      0.71     48308
           0       0.00      0.00      0.00         0
           1       0.83      0.37      0.51     51484

   micro avg       0.64      0.64      0.64     99792
   macro avg       0.47      0.43      0.41     99792
weighted avg       0.71      0.64      0.61     99792


--- Generating Future Forecast ---
Forecast generated for the next 3 hours.
            timestamp  Predicted_Reading
0 2025-10-09 00:00:00          50.898290
1 2025-10-09 00:00:05          62.272171
2 2025-10-09 00:00:10          62.334851
3 2025-10-09 00:00:15          62.334851
4 2025-10-09 00:00:20          62.272171
Forecast saved to future_predictions_generated.csv
Forecast generated for the next 3 hours.
            timestamp  Predicted_Reading
0 2025-10-09 00:00:00          50.898290
1 2025-10-09 00:00:05          62.272171
2 2025-10-09 00:00:10          62.334851
3 2025-10-09 00:00:15          62.334851

In [6]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]




[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
